# RT Notebook 24 — D/E Metamorphic Independence Test

This notebook independently tests the frozen bounded P127/P128 candidate semantics through **metamorphic relations** rather than by replaying Notebook 23's labeled fixture matrix.

**Claim ceiling:** `C2_LIMITATION_OR_NEGATIVE_RESULT`

A clean run supports only bounded consistency with the declared transformations. Any mismatch is preserved as counterexample evidence.


In [ ]:
from __future__ import annotations
import copy, hashlib, json, random
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Mapping, Sequence

NOTEBOOK_ID = "RT_Notebook_24_D_E_Metamorphic_Independence"
SPEC_ID = "MPF_SIM_D_E_METAMORPHIC_INDEPENDENCE_001"
ROOT = Path.cwd()
RESULT_DIR = ROOT / "departments/colab/results" / SPEC_ID
try:
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
except PermissionError:
    ROOT = Path("/tmp")
    RESULT_DIR = ROOT / "departments/colab/results" / SPEC_ID
    RESULT_DIR.mkdir(parents=True, exist_ok=True)

def canonical_bytes(v: Any) -> bytes:
    return json.dumps(v, sort_keys=True, separators=(",",":"), ensure_ascii=False, allow_nan=False).encode("utf-8")
def sha256_json(v: Any) -> str:
    return hashlib.sha256(canonical_bytes(v)).hexdigest()
def token(*parts: Any, length: int=20) -> str:
    return hashlib.sha256("|".join(map(str,parts)).encode()).hexdigest()[:length]
print("Result directory:", RESULT_DIR.resolve())


## 1. Embedded independent specification

In [ ]:
EXPERIMENT_SPEC = {'spec_id': 'MPF_SIM_D_E_METAMORPHIC_INDEPENDENCE_001', 'schema_version': '1.0.0', 'status': 'IMMUTABLE_SPEC', 'created_at': '2026-07-30T20:32:00-04:00', 'research_question': 'Do frozen bounded P127/P128 candidate predicates satisfy independently declared metamorphic invariants under novel record construction and single-field transformations?', 'independence_basis': ['No reuse of Notebook 23 held-out fixture rows or its expected-control matrix.', 'Novel seeds 719, 823, 947 and independently constructed payloads.', 'Expected outcomes derived from metamorphic relations and a separate declarative oracle, not source-relation case labels.', 'Single-field mutations isolate causal dependencies.', 'Input order is independently shuffled before a second replay.'], 'subject_under_test': 'P127_P128_CANDIDATE_20260730_001', 'contexts': ['C4', 'C5', 'C6'], 'profiles': ['low', 'mid', 'high'], 'seeds': [719, 823, 947], 'metamorphic_relations': [{'id': 'MR01', 'name': 'valid_baseline_acceptance', 'expectation': 'A fully bound valid record is REPRESENTABLE and NON_COLLAPSED.'}, {'id': 'MR02', 'name': 'witness_removal_localization', 'expectation': 'Removing witness changes P127 to REJECT_WITNESS and leaves P128 unchanged.'}, {'id': 'MR03', 'name': 'history_removal_localization', 'expectation': 'Removing history changes P127 to REJECT_HISTORY and leaves P128 unchanged.'}, {'id': 'MR04', 'name': 'type_mutation_localization', 'expectation': 'Changing relation type changes P127 to REJECT_TYPE and leaves P128 unchanged.'}, {'id': 'MR05', 'name': 'context_nontransport', 'expectation': 'Changing source context changes P127 to REJECT_CONTEXT and P128 to REJECT_PROFILE.'}, {'id': 'MR06', 'name': 'threshold_boundary_strictness', 'expectation': 'Setting distinction equal to epsilon leaves P127 unchanged and changes P128 to REJECT_SUBTHRESHOLD.'}, {'id': 'MR07', 'name': 'threshold_monotonicity', 'expectation': 'For fixed valid bindings, distinction below/equal threshold rejects and above threshold admits.'}, {'id': 'MR08', 'name': 'witness_enrichment_invariance', 'expectation': 'Bare versus enriched complete witness does not alter classifications.'}, {'id': 'MR09', 'name': 'history_extension_invariance', 'expectation': 'Appending valid history entries does not alter classifications.'}, {'id': 'MR10', 'name': 'irrelevant_payload_permutation_invariance', 'expectation': 'Changing trace and payload tokens without changing required bindings does not alter classifications.'}], 'claim_ceiling': 'C2_LIMITATION_OR_NEGATIVE_RESULT', 'blocked_interpretations': ['Universal D/E semantics', 'Theorem promotion', 'Injectivity', 'Reversibility', 'Physical validation'], 'content_sha256': 'a31217df001f218f8aefb3b204d18322a65333b82489abe7de4b53ddf26c1dae'}
assert sha256_json({**EXPERIMENT_SPEC, 'content_sha256': ''}) == EXPERIMENT_SPEC['content_sha256']
print(json.dumps(EXPERIMENT_SPEC, indent=2))

## 2. Frozen subject under test

These functions are the tested candidate predicates. The independent oracle below does not call them and does not share their control-flow implementation.


In [ ]:
VALID_THRESHOLD_PROFILES=("low","mid","high")
THRESHOLDS={
 "C4":{"low":0.20,"mid":0.50,"high":0.80},
 "C5":{"low":0.25,"mid":0.55,"high":0.85},
 "C6":{"low":0.30,"mid":0.60,"high":0.90},
}
FROZEN_PREDICATE_VERSION="P127_P128_CANDIDATE_20260730_001"

def classify_representability(row: Mapping[str,Any]) -> Dict[str,str]:
    if row["threshold_profile"] not in VALID_THRESHOLD_PROFILES:
        return {"classification":"REJECT_PROFILE","reason":"unknown_threshold_profile"}
    if row["relation_type"] != row["required_relation_type"]:
        return {"classification":"REJECT_TYPE","reason":"source_relation_type_mismatch"}
    if row["source_context"] != row["evaluation_context"]:
        return {"classification":"REJECT_CONTEXT","reason":"cross_context_source_relation"}
    if not row["witness_present"] or not row["witness_complete"]:
        return {"classification":"REJECT_WITNESS","reason":"missing_or_incomplete_typed_witness"}
    if not row["history_present"]:
        return {"classification":"REJECT_HISTORY","reason":"missing_source_relation_history"}
    return {"classification":"REPRESENTABLE","reason":"all_bounded_requirements_satisfied"}

def classify_noncollapse(row: Mapping[str,Any]) -> Dict[str,str]:
    profile=row["threshold_profile"]; context=row["evaluation_context"]
    if profile not in VALID_THRESHOLD_PROFILES or context not in THRESHOLDS:
        return {"classification":"REJECT_PROFILE","reason":"unknown_context_or_threshold_profile"}
    if row["source_context"] != context:
        return {"classification":"REJECT_PROFILE","reason":"threshold_profile_not_transportable_across_context"}
    if row["distinction"] <= 0.0:
        return {"classification":"REJECT_DISTINCTION","reason":"nonpositive_distinction"}
    if row["distinction"] <= THRESHOLDS[context][profile]:
        return {"classification":"REJECT_SUBTHRESHOLD","reason":"distinction_not_strictly_above_context_threshold"}
    return {"classification":"NON_COLLAPSED","reason":"distinction_strictly_above_context_threshold"}

SUBJECT_HASH=sha256_json({"version":FROZEN_PREDICATE_VERSION,"thresholds":THRESHOLDS})
print("Subject declaration hash:",SUBJECT_HASH)


## 3. Independent declarative oracle

In [ ]:
# The oracle is expressed as data requirements plus priority tables.
REP_PRIORITY=[
 ("REJECT_PROFILE", lambda r: r.get("threshold_profile") not in {"low","mid","high"}),
 ("REJECT_TYPE", lambda r: r.get("relation_type") != r.get("required_relation_type")),
 ("REJECT_CONTEXT", lambda r: r.get("source_context") != r.get("evaluation_context")),
 ("REJECT_WITNESS", lambda r: not (r.get("witness_present") and r.get("witness_complete"))),
 ("REJECT_HISTORY", lambda r: not r.get("history_present")),
]
def oracle_rep(r):
    return next((label for label,pred in REP_PRIORITY if pred(r)),"REPRESENTABLE")

def oracle_non(r):
    p=r.get("threshold_profile"); c=r.get("evaluation_context")
    if p not in {"low","mid","high"} or c not in THRESHOLDS: return "REJECT_PROFILE"
    if r.get("source_context") != c: return "REJECT_PROFILE"
    d=r.get("distinction")
    if d <= 0: return "REJECT_DISTINCTION"
    return "NON_COLLAPSED" if d > THRESHOLDS[c][p] else "REJECT_SUBTHRESHOLD"

def observe(r):
    return {
      "representability":classify_representability(r)["classification"],
      "noncollapse":classify_noncollapse(r)["classification"],
    }
def oracle(r):
    return {"representability":oracle_rep(r),"noncollapse":oracle_non(r)}


## 4. Novel baseline construction

In [ ]:
CONTEXTS=("C4","C5","C6"); PROFILES=("low","mid","high"); SEEDS=(719,823,947)

def make_baseline(context,profile,seed,witness_mode):
    eps=THRESHOLDS[context][profile]
    # Distinction is independently generated but guaranteed strictly above epsilon.
    room=max(0.01,0.98-eps)
    offset=0.01+(int(token("offset",context,profile,seed,witness_mode,length=8),16)%10000)/10000*room
    distinction=min(0.999999,eps+offset)
    rid=token("N24",context,profile,seed,witness_mode,length=24)
    return {
      "row_id":rid,"evaluation_context":context,"source_context":context,
      "threshold_profile":profile,"relation_type":"D_SOURCE_RELATION_C",
      "required_relation_type":"D_SOURCE_RELATION_C",
      "witness_present":True,"witness_complete":True,
      "witness_mode":witness_mode,
      "witness_token":{"token":token(rid,"witness"),"mode":witness_mode,
          "relation_type":"D_SOURCE_RELATION_C" if witness_mode=="enriched" else None,
          "context":context if witness_mode=="enriched" else None},
      "history_present":True,"history":[token(rid,"h",i) for i in range(2)],
      "trace":[token(rid,"t",i) for i in range(5)],
      "ordered_source_payload":[token(rid,"p",i) for i in range(4)],
      "distinction":round(distinction,12),"epsilon_a_C":eps,"seed":seed,
    }

BASELINES=[make_baseline(c,p,s,w) for c in CONTEXTS for p in PROFILES for s in SEEDS for w in ("bare","enriched")]
assert len(BASELINES)==54 and len({r["row_id"] for r in BASELINES})==54
assert all(observe(r)==oracle(r)=={"representability":"REPRESENTABLE","noncollapse":"NON_COLLAPSED"} for r in BASELINES)
print("Novel valid baselines:",len(BASELINES))


## 5. Metamorphic transformations

In [ ]:
def mutate(base, relation_id):
    r=copy.deepcopy(base)
    if relation_id=="MR01": pass
    elif relation_id=="MR02":
        r["witness_present"]=False; r["witness_complete"]=False; r["witness_token"]=None
    elif relation_id=="MR03":
        r["history_present"]=False; r["history"]=[]
    elif relation_id=="MR04":
        r["relation_type"]="D_UNRELATED_RELATION_C"
    elif relation_id=="MR05":
        r["source_context"]=CONTEXTS[(CONTEXTS.index(r["evaluation_context"])+1)%3]
    elif relation_id=="MR06":
        r["distinction"]=r["epsilon_a_C"]
    elif relation_id=="MR08":
        new_mode="enriched" if r["witness_mode"]=="bare" else "bare"
        r["witness_mode"]=new_mode
        r["witness_token"]["mode"]=new_mode
        r["witness_token"]["relation_type"]="D_SOURCE_RELATION_C" if new_mode=="enriched" else None
        r["witness_token"]["context"]=r["source_context"] if new_mode=="enriched" else None
    elif relation_id=="MR09":
        r["history"].extend([token(r["row_id"],"extra-history",i) for i in range(3)])
    elif relation_id=="MR10":
        r["trace"]=list(reversed(r["trace"]))+[token(r["row_id"],"new-trace")]
        r["ordered_source_payload"]=[token(r["row_id"],"alternate",i) for i in range(6)]
    else: raise ValueError(relation_id)
    r["mutation_id"]=relation_id
    r["mutant_id"]=token(r["row_id"],relation_id,length=24)
    return r

EXPECTED_TRANSITION={
 "MR01":("REPRESENTABLE","NON_COLLAPSED"),
 "MR02":("REJECT_WITNESS","NON_COLLAPSED"),
 "MR03":("REJECT_HISTORY","NON_COLLAPSED"),
 "MR04":("REJECT_TYPE","NON_COLLAPSED"),
 "MR05":("REJECT_CONTEXT","REJECT_PROFILE"),
 "MR06":("REPRESENTABLE","REJECT_SUBTHRESHOLD"),
 "MR08":("REPRESENTABLE","NON_COLLAPSED"),
 "MR09":("REPRESENTABLE","NON_COLLAPSED"),
 "MR10":("REPRESENTABLE","NON_COLLAPSED"),
}
MUTANTS=[mutate(b,mr) for b in BASELINES for mr in EXPECTED_TRANSITION]
print("Single-transform mutants:",len(MUTANTS))


## 6. Execute differential and metamorphic checks

In [ ]:
RECORDS=[]
for r in MUTANTS:
    obs=observe(r); exp=oracle(r)
    declared=dict(zip(("representability","noncollapse"),EXPECTED_TRANSITION[r["mutation_id"]]))
    RECORDS.append({
      "mutant_id":r["mutant_id"],"base_row_id":r["row_id"],"relation_id":r["mutation_id"],
      "observed":obs,"oracle":exp,"declared_transition":declared,
      "subject_oracle_agreement":obs==exp,
      "metamorphic_agreement":obs==declared,
      "input_sha256":sha256_json(r),
    })

# MR07 is a three-point monotonic threshold test.
for b in BASELINES:
    eps=b["epsilon_a_C"]
    points=[("below",max(1e-12,eps-1e-9),"REJECT_SUBTHRESHOLD"),
            ("equal",eps,"REJECT_SUBTHRESHOLD"),
            ("above",eps+1e-9,"NON_COLLAPSED")]
    observed_sequence=[]
    for position,value,expected_non in points:
        r=copy.deepcopy(b); r["distinction"]=value
        obs=observe(r); observed_sequence.append(obs["noncollapse"])
        RECORDS.append({
          "mutant_id":token(b["row_id"],"MR07",position,length=24),
          "base_row_id":b["row_id"],"relation_id":"MR07","threshold_position":position,
          "observed":obs,"oracle":oracle(r),
          "declared_transition":{"representability":"REPRESENTABLE","noncollapse":expected_non},
          "subject_oracle_agreement":obs==oracle(r),
          "metamorphic_agreement":obs=={"representability":"REPRESENTABLE","noncollapse":expected_non},
          "input_sha256":sha256_json(r),
        })
    assert observed_sequence==["REJECT_SUBTHRESHOLD","REJECT_SUBTHRESHOLD","NON_COLLAPSED"]

print("Total checks:",len(RECORDS))


## 7. Independent order-shuffled replay

In [ ]:
def replay_digest(records):
    normalized=sorted(records,key=lambda x:x["mutant_id"])
    return sha256_json(normalized)

PASS_A=copy.deepcopy(RECORDS)
PASS_B=copy.deepcopy(RECORDS)
random.Random(1907).shuffle(PASS_B)
REPLAY_A_SHA256=replay_digest(PASS_A)
REPLAY_B_SHA256=replay_digest(PASS_B)
REPLAY_AGREEMENT=REPLAY_A_SHA256==REPLAY_B_SHA256
print("Replay agreement:",REPLAY_AGREEMENT,REPLAY_A_SHA256)


## 8. Preserve flags and summarize

In [ ]:
FLAGS=[r for r in RECORDS if not (r["subject_oracle_agreement"] and r["metamorphic_agreement"])]
if not REPLAY_AGREEMENT:
    FLAGS.append({"relation_id":"REPLAY","preservation_status":"PRESERVED_COUNTEREXAMPLE_OR_LIMITATION",
                  "pass_a_sha256":REPLAY_A_SHA256,"pass_b_sha256":REPLAY_B_SHA256})
counts=Counter(r["relation_id"] for r in RECORDS)
passes=Counter(r["relation_id"] for r in RECORDS if r["subject_oracle_agreement"] and r["metamorphic_agreement"])
ALL_PASS=(not FLAGS) and REPLAY_AGREEMENT
SUMMARY={
 "spec_id":SPEC_ID,"notebook_id":NOTEBOOK_ID,
 "status":"BOUNDED_SUPPORT" if ALL_PASS else "LIMITATION_OR_NEGATIVE_RESULT",
 "claim_ceiling":EXPERIMENT_SPEC["claim_ceiling"],
 "generated_at_utc":datetime.now(timezone.utc).isoformat(),
 "baseline_count":len(BASELINES),"check_count":len(RECORDS),"flag_count":len(FLAGS),
 "replay_agreement":REPLAY_AGREEMENT,
 "checks_by_relation":dict(sorted(counts.items())),
 "passes_by_relation":dict(sorted(passes.items())),
 "independence_basis":EXPERIMENT_SPEC["independence_basis"],
 "bounded_conclusion":(
   "All independently declared metamorphic relations and the separate declarative oracle agreed "
   "for the finite generated records." if ALL_PASS else
   "One or more independently declared metamorphic relations, oracle comparisons, or replay checks failed; "
   "the evidence is preserved and blocks bounded support."
 ),
 "blocked_interpretations":EXPERIMENT_SPEC["blocked_interpretations"],
}
FALSIFICATION_REPORT={"spec_id":SPEC_ID,"status":"NO_FLAGS" if not FLAGS else "PRESERVED_FLAGS_PRESENT",
 "flag_count":len(FLAGS),"flags":FLAGS,
 "negative_evidence_policy":"No mismatch may be discarded or relabeled post hoc."}
print(json.dumps(SUMMARY,indent=2))


## 9. Write artifacts and manifest

In [ ]:
def write_json(path,value):
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,ensure_ascii=False)+"\n",encoding="utf-8")
def write_jsonl(path,rows):
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open("w",encoding="utf-8",newline="\n") as f:
        for row in rows: f.write(json.dumps(row,sort_keys=True,separators=(",",":"),ensure_ascii=False)+"\n")

records_path=RESULT_DIR/"metamorphic_records.jsonl"
summary_path=RESULT_DIR/"summary.json"
flags_path=RESULT_DIR/"falsification_report.json"
spec_path=RESULT_DIR/"experiment_spec.json"
manifest_path=RESULT_DIR/"manifest.json"
write_jsonl(records_path,RECORDS); write_json(summary_path,SUMMARY)
write_json(flags_path,FALSIFICATION_REPORT); write_json(spec_path,EXPERIMENT_SPEC)
artifact_hashes={str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest()
                 for p in (records_path,summary_path,flags_path,spec_path)}
MANIFEST={"spec_id":SPEC_ID,"notebook_id":NOTEBOOK_ID,
 "generated_at_utc":datetime.now(timezone.utc).isoformat(),
 "subject_version":FROZEN_PREDICATE_VERSION,"subject_declaration_sha256":SUBJECT_HASH,
 "experiment_spec_content_sha256":EXPERIMENT_SPEC["content_sha256"],
 "pass_a_sha256":REPLAY_A_SHA256,"pass_b_sha256":REPLAY_B_SHA256,
 "artifact_hashes":artifact_hashes,
 "manifest_hash_policy":"SHA-256 over exact written bytes; manifest excludes its own hash."}
write_json(manifest_path,MANIFEST)
for p in (records_path,summary_path,flags_path,spec_path,manifest_path):
    print(p.resolve(),p.stat().st_size,"bytes")


## 10. Final gates

In [ ]:
required=[RESULT_DIR/n for n in ("metamorphic_records.jsonl","summary.json","falsification_report.json","experiment_spec.json","manifest.json")]
assert all(p.exists() and p.stat().st_size>0 for p in required)
assert REPLAY_AGREEMENT
assert not FLAGS, "Independent test flags were preserved in falsification_report.json"
assert all(r["subject_oracle_agreement"] and r["metamorphic_agreement"] for r in RECORDS)
print("PASS: independent metamorphic D/E test completed with no flagged contradictions.")
print("Claim remains bounded by:",EXPERIMENT_SPEC["claim_ceiling"])


## Interpretation

A clean result establishes only finite, bounded agreement between the frozen candidate predicates, an independently expressed declarative oracle, and the declared metamorphic transformations. It does not establish universal D/E semantics or theorem closure.
